# Lecture 1 author-facing figure notebook: task examples

This notebook generates the three figures used in the new Lecture 1 sections
"From inference to machine-learning problems" and "What do we use learned
models for?" in `slide_deck.typ`. It follows the same course figure
convention as `coin_model.ipynb`:

- the notebook lives in the lecture directory;
- generated assets go in `figures/`;
- each filename is `<notebook-stem>__<figure-name>.<ext>`;
- the figure path is shown on the slide as provenance.

## Provenance

All three figures are lecture adaptations of figures from Kevin Murphy,
*Probabilistic Machine Learning: Advanced Topics*, built by reproducing the
underlying idea from the book's public code rather than screenshotting the
PDF.

| Task | Murphy figure | Original ProbML notebook | What we did |
|---|---|---|---|
| Prediction | Fig. 15.8 (Ch. 15, logistic regression) | `logreg_iris_bayes_2d.ipynb` | Reproduced directly: same Iris petal-length/petal-width, versicolor-vs-virginica 2D subset; refit a Bayesian logistic regression locally (MAP + Laplace approximation) and plotted the posterior-predictive heatmap with a fan of sampled decision boundaries, matching the figure's visual argument. |
| Generation | Fig. 21.2 (Ch. 21, VAEs) | `celeba_vae_ae_comparison.ipynb` | **Adapted, not reproduced at scale.** The original trains a convolutional VAE on CelebA (~200k, 64x64 RGB face crops) on a GPU. For a fast, dependency-light, CPU-only lecture build we substitute scikit-learn's bundled Olivetti faces dataset (400, 64x64 grayscale faces) and a small fully-connected VAE trained for ~10 seconds on CPU. This keeps the conceptual structure identical (`z ~ p(z)`, `x \| z ~ p_theta(x \| z)`, sampling as the generative operation) while sacrificing photorealism — the blurriness of the samples is itself a genuine and pedagogically useful property of pixel-wise VAE likelihoods, not an artifact of the adaptation. |
| Discovery | Fig. 28.2 (Ch. 28, mixture models) | `gmm_2d.ipynb` | Reproduced the structure directly: synthetic unlabeled 2D data from three Gaussian components with distinct (non-spherical) covariance, fit with `sklearn.mixture.GaussianMixture`, and plotted alongside the unlabeled data to make the "unlabeled data -> inferred latent grouping" argument visually explicit. |

Deterministic seeds are set throughout so the notebook reproduces the same
figures on every run.


In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

# Reproducibility
RNG = np.random.default_rng(20260915)

# Works both locally and in Colab.
OUT = Path("figures")
OUT.mkdir(exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.bbox": "tight",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})

def save_fig(fig, name, **kwargs):
    path = OUT / name
    fig.savefig(path, bbox_inches="tight", **kwargs)
    print(f"saved {path}")
    return path

## 1. Prediction: Bayesian logistic regression on Iris

Given measured flower features $x_*$, what should we say about the class
$y_*$? We fit $p(y \mid x, \theta)$ by logistic regression, put a Gaussian
prior on $\theta$, and approximate the posterior with a Laplace
approximation (MAP estimate plus the inverse Hessian of the negative log
posterior at the MAP). The predictive object is the full distribution
$p(y_* \mid x_*, D) = \int p(y_* \mid x_*, \theta)\, p(\theta \mid D)\,
d\theta$, approximated here by averaging over posterior samples of
$\theta$ — not a single point estimate.

In [ ]:
from sklearn.datasets import load_iris
from scipy.optimize import minimize

iris = load_iris()
X_all, y_all = iris.data, iris.target

# Matches Murphy Fig 15.8: petal length & width, versicolor vs virginica.
mask = y_all != 0
X = X_all[mask][:, [2, 3]]
y = (y_all[mask] - 1).astype(float)  # 0 = versicolor, 1 = virginica

mu_x, sd_x = X.mean(0), X.std(0)
Xs = (X - mu_x) / sd_x
Phi = np.column_stack([np.ones(len(Xs)), Xs])

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

prior_precision = 1.0  # Gaussian prior N(0, 1/prior_precision * I) on weights

def neg_log_posterior(w):
    z = Phi @ w
    ll = np.sum(y * z - np.log1p(np.exp(z)))
    lp = -0.5 * prior_precision * np.sum(w ** 2)
    return -(ll + lp)

def grad_neg_log_posterior(w):
    p = sigmoid(Phi @ w)
    grad_ll = Phi.T @ (y - p)
    grad_lp = -prior_precision * w
    return -(grad_ll + grad_lp)

result = minimize(neg_log_posterior, np.zeros(3), jac=grad_neg_log_posterior, method="BFGS")
w_map = result.x

# Laplace approximation: Gaussian centered at the MAP with covariance given
# by the inverse Hessian of the negative log posterior.
p_map = sigmoid(Phi @ w_map)
hessian = Phi.T @ np.diag(p_map * (1 - p_map)) @ Phi + prior_precision * np.eye(3)
cov = np.linalg.inv(hessian)

print("MAP weights:", w_map)
print("Posterior standard deviations:", np.sqrt(np.diag(cov)))

In [ ]:
x1 = np.linspace(Xs[:, 0].min() - 0.5, Xs[:, 0].max() + 0.5, 150)
x2 = np.linspace(Xs[:, 1].min() - 0.5, Xs[:, 1].max() + 0.5, 150)
G1, G2 = np.meshgrid(x1, x2)
grid = np.column_stack([np.ones(G1.size), G1.ravel(), G2.ravel()])

n_samples = 40
w_samples = RNG.multivariate_normal(w_map, cov, size=n_samples)
probs_samples = sigmoid(grid @ w_samples.T)
prob_mean = probs_samples.mean(axis=1).reshape(G1.shape)

fig, ax = plt.subplots(figsize=(7, 5.5))
cf = ax.contourf(G1, G2, prob_mean, levels=20, cmap="RdBu_r", alpha=0.75, vmin=0, vmax=1)
for s in range(n_samples):
    probs = sigmoid(grid @ w_samples[s]).reshape(G1.shape)
    ax.contour(G1, G2, probs, levels=[0.5], colors="k", linewidths=0.6, alpha=0.35)
ax.contour(G1, G2, prob_mean, levels=[0.5], colors="k", linewidths=2.2)
ax.scatter(Xs[y == 0, 0], Xs[y == 0, 1], c="C0", edgecolor="k", s=45, label="versicolor")
ax.scatter(Xs[y == 1, 0], Xs[y == 1, 1], c="C3", edgecolor="k", s=45, label="virginica")
ax.set(
    xlabel="petal length (standardized)",
    ylabel="petal width (standardized)",
    title="Bayesian logistic regression: posterior decision boundaries",
)
ax.legend(frameon=False, loc="upper left")
fig.colorbar(cf, ax=ax, label=r"$p(y=\mathrm{virginica}\mid x)$")
fig.tight_layout()

save_fig(fig, "task_examples_figures__prediction_iris_logistic.svg")
plt.show()

## 2. Generation: sampling new faces from a learned latent-variable model

A generative task asks the model to produce new, plausible objects, not
just a summary of an existing one. We train a small variational
autoencoder: $z \sim p(z) = \mathcal{N}(0, I)$, $x \mid z \sim
p_\theta(x \mid z)$, learned by maximizing an evidence lower bound. New
faces are produced by sampling $z^{(s)} \sim p(z)$ and decoding:
$x^{(s)} \sim p_\theta(x \mid z^{(s)})$.

See the provenance note above: this is a lightweight CPU adaptation of
Murphy's CelebA VAE figure using the much smaller Olivetti faces dataset,
so treat the samples as illustrating the *mechanism*, not as a
photorealism benchmark.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.datasets import fetch_olivetti_faces

torch.manual_seed(20260915)

faces = fetch_olivetti_faces(shuffle=True, random_state=20260915)
X_faces = faces.images.astype(np.float32)  # (400, 64, 64), already in [0, 1]
n_faces, H, W = X_faces.shape
X_faces_t = torch.from_numpy(X_faces.reshape(n_faces, -1))

D, LATENT_DIM, HIDDEN_DIM = H * W, 16, 256


class SmallVAE(nn.Module):
    'A deliberately small, fully-connected VAE for fast CPU training.'

    def __init__(self):
        super().__init__()
        self.enc1 = nn.Linear(D, HIDDEN_DIM)
        self.enc_mu = nn.Linear(HIDDEN_DIM, LATENT_DIM)
        self.enc_logvar = nn.Linear(HIDDEN_DIM, LATENT_DIM)
        self.dec1 = nn.Linear(LATENT_DIM, HIDDEN_DIM)
        self.dec2 = nn.Linear(HIDDEN_DIM, D)

    def encode(self, x):
        h = F.relu(self.enc1(x))
        return self.enc_mu(h), self.enc_logvar(h)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        return mu + torch.randn_like(std) * std

    def decode(self, z):
        h = F.relu(self.dec1(z))
        return torch.sigmoid(self.dec2(h))

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        return self.decode(z), mu, logvar


def vae_loss(recon, x, mu, logvar):
    reconstruction = F.binary_cross_entropy(recon, x, reduction="sum")
    kl_divergence = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return reconstruction + kl_divergence


model = SmallVAE()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

n_epochs, batch_size = 150, 64
for epoch in range(n_epochs):
    perm = torch.randperm(n_faces)
    for i in range(0, n_faces, batch_size):
        batch = X_faces_t[perm[i : i + batch_size]]
        optimizer.zero_grad()
        recon, mu, logvar = model(batch)
        loss = vae_loss(recon, batch, mu, logvar)
        loss.backward()
        optimizer.step()

print("training complete")

In [ ]:
model.eval()
with torch.no_grad():
    real_idx = RNG.choice(n_faces, size=8, replace=False)
    real_imgs = X_faces_t[real_idx].numpy().reshape(-1, H, W)
    z_samples = torch.randn(8, LATENT_DIM)
    gen_imgs = model.decode(z_samples).numpy().reshape(-1, H, W)

fig, axes = plt.subplots(2, 8, figsize=(12, 3.4))
for i in range(8):
    axes[0, i].imshow(real_imgs[i], cmap="gray", vmin=0, vmax=1)
    axes[1, i].imshow(gen_imgs[i], cmap="gray", vmin=0, vmax=1)
    for row in (0, 1):
        axes[row, i].set_xticks([])
        axes[row, i].set_yticks([])
        for spine in axes[row, i].spines.values():
            spine.set_visible(False)

axes[0, 0].set_ylabel("training\ndata", rotation=0, ha="right", va="center")
axes[1, 0].set_ylabel(r"$x^{(s)}\!\sim p_\theta(x\mid z^{(s)})$", rotation=0, ha="right", va="center")
fig.suptitle("Generation: sampling new faces from a learned latent-variable model", y=1.02)
fig.tight_layout()

save_fig(fig, "task_examples_figures__generation_vae_samples.png", dpi=150)
plt.show()

## 3. Discovery: inferring latent structure with a Gaussian mixture

Here we are not predicting a supplied label, nor synthesizing a new
observation. We are inferring hidden structure — which group each point
belongs to — that organizes unlabeled observations:
$z \sim p(z)$, $x \mid z \sim p(x \mid z)$, and after observing $x$ we
compute the posterior $p(z \mid x)$ over cluster membership. Note that a
Gaussian mixture model is, formally, a generative model of $x$ — but the
teaching point here (as in Murphy Ch. 28) is that the *task* we are using
it for is discovery: recovering meaningful latent variables, not sampling
new realistic-looking data.

In [ ]:
from matplotlib.patches import Ellipse
from sklearn.mixture import GaussianMixture

# Three elliptical clusters with distinct covariance, unlabeled.
means = [(-2.5, 1.0), (2.0, 2.0), (0.0, -2.5)]
covariances = [
    [[0.6, 0.35], [0.35, 0.4]],
    [[0.5, -0.3], [-0.3, 0.9]],
    [[1.0, 0.0], [0.0, 0.25]],
]
sizes = [110, 90, 130]

X_parts, z_true_parts = [], []
for k, (mean, cov, size) in enumerate(zip(means, covariances, sizes)):
    X_parts.append(RNG.multivariate_normal(mean, cov, size=size))
    z_true_parts.append(np.full(size, k))
X_clusters = np.vstack(X_parts)
perm = RNG.permutation(len(X_clusters))
X_clusters = X_clusters[perm]

gmm = GaussianMixture(n_components=3, covariance_type="full", random_state=0, n_init=5)
z_hat = gmm.fit_predict(X_clusters)


def draw_covariance_ellipse(ax, mean, cov, color, n_std=2.0):
    vals, vecs = np.linalg.eigh(cov)
    order = vals.argsort()[::-1]
    vals, vecs = vals[order], vecs[:, order]
    angle = np.degrees(np.arctan2(*vecs[:, 0][::-1]))
    width, height = 2 * n_std * np.sqrt(vals)
    ax.add_patch(Ellipse(mean, width, height, angle=angle, facecolor="none",
                          edgecolor=color, linewidth=2.2))


fig, axes = plt.subplots(1, 2, figsize=(11, 4.6))

axes[0].scatter(X_clusters[:, 0], X_clusters[:, 1], s=22, c="0.35", alpha=0.75)
axes[0].set(title="Observed data (unlabeled)", xlabel=r"$x_1$", ylabel=r"$x_2$")

colors = ["C0", "C1", "C2"]
order = np.argsort(gmm.means_[:, 0])
color_for = {component: colors[i] for i, component in enumerate(order)}
point_colors = [color_for[k] for k in z_hat]
axes[1].scatter(X_clusters[:, 0], X_clusters[:, 1], s=22, c=point_colors, alpha=0.8)
for k in range(3):
    draw_covariance_ellipse(axes[1], gmm.means_[k], gmm.covariances_[k], color_for[k])
    axes[1].scatter(*gmm.means_[k], marker="x", s=90, c="k", linewidths=2.2)
axes[1].set(title=r"Inferred latent structure: $p(z \mid x)$", xlabel=r"$x_1$", ylabel=r"$x_2$")

for ax in axes:
    ax.set_xlim(-5.5, 5)
    ax.set_ylim(-4.5, 4.5)
    ax.set_aspect("equal")

fig.tight_layout()
save_fig(fig, "task_examples_figures__discovery_gmm_clusters.svg")
plt.show()

## Figure manifest

`slide_deck.typ` expects these files:

- `figures/task_examples_figures__prediction_iris_logistic.svg`
- `figures/task_examples_figures__generation_vae_samples.png`
- `figures/task_examples_figures__discovery_gmm_clusters.svg`

This notebook adds `scikit-learn` and `torch` (CPU build) to the project
environment, on top of the NumPy/SciPy/Matplotlib baseline used by
`coin_model.ipynb`. Both are pinned in `pyproject.toml`.